# Limpeza e Preparação dos Dados

Este notebook lê os CSVs brutos, aplica todas as transformações de limpeza e exporta o resultado em Parquet para uso nos notebooks de análise e modelagem.

**Pipeline:**
1. Joins com airlines e airports
2. Remoção de colunas sem valor preditivo
3. Remoção de voos cancelados
4. Tratamento de NaN nas causas de atraso
5. Remoção de registros sem DEPARTURE_DELAY
6. Tratamento de NaN nas colunas de join
7. Feature engineering
8. Exportação em Parquet

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
flights = pd.read_csv("../data/flights.csv", low_memory=False)
airports = pd.read_csv("../data/airports.csv")
airlines = pd.read_csv("../data/airlines.csv")

# Join airlines
flights = flights.merge(airlines, left_on="AIRLINE", right_on="IATA_CODE", how="left") \
                 .drop(columns="IATA_CODE") \
                 .rename(columns={"AIRLINE_y": "AIRLINE_NAME", "AIRLINE_x": "AIRLINE"})

# Join airports — ORIGIN
flights = flights.merge(airports, left_on="ORIGIN_AIRPORT", right_on="IATA_CODE", how="left") \
                 .drop(columns="IATA_CODE") \
                 .rename(columns={
                     "AIRPORT": "ORIGIN_AIRPORT_NAME",
                     "CITY": "ORIGIN_CITY",
                     "STATE": "ORIGIN_STATE",
                     "LATITUDE": "ORIGIN_LAT",
                     "LONGITUDE": "ORIGIN_LON"
                 }).drop(columns=["COUNTRY"])

# Join airports — DESTINATION
flights = flights.merge(airports, left_on="DESTINATION_AIRPORT", right_on="IATA_CODE", how="left") \
                 .drop(columns="IATA_CODE") \
                 .rename(columns={
                     "AIRPORT": "DEST_AIRPORT_NAME",
                     "CITY": "DEST_CITY",
                     "STATE": "DEST_STATE",
                     "LATITUDE": "DEST_LAT",
                     "LONGITUDE": "DEST_LON"
                 }).drop(columns=["COUNTRY"])

print(f"Shape inicial (após joins): {flights.shape}")

Shape inicial (após joins): (5819079, 42)


## Remoção de colunas sem valor preditivo

- **YEAR**: constante (2015), variância zero
- **TAIL_NUMBER**: identificador da aeronave (4.897 valores únicos), sem valor preditivo
- **FLIGHT_NUMBER**: número do voo (6.952 valores únicos), sem valor preditivo

In [3]:
flights = flights.drop(columns=["YEAR", "TAIL_NUMBER", "FLIGHT_NUMBER"])
print(f"Shape após drop de colunas: {flights.shape}")

Shape após drop de colunas: (5819079, 39)


## Remoção de voos cancelados

Voos cancelados **não são voos atrasados** — são uma categoria diferente. Mantê-los confundiria a modelagem de atrasos, pois não possuem DEPARTURE_DELAY real.

Após a remoção, as colunas `CANCELLED` (sempre 0) e `CANCELLATION_REASON` (sempre NaN) tornam-se constantes e também são removidas.

In [4]:
n_before = len(flights)
n_cancelled = flights['CANCELLED'].sum()
flights = flights.query("CANCELLED == 0").drop(columns=["CANCELLED", "CANCELLATION_REASON"])
print(f"Voos cancelados removidos: {n_cancelled:,}")
print(f"Shape após remoção: {flights.shape}")

Voos cancelados removidos: 89,884
Shape após remoção: (5729195, 37)


## Tratamento de colunas de causa de atraso

As 5 colunas de causa (`AIR_SYSTEM_DELAY`, `SECURITY_DELAY`, `AIRLINE_DELAY`, `LATE_AIRCRAFT_DELAY`, `WEATHER_DELAY`) possuem ~82% de NaN **por design** — só são preenchidas quando o voo é atrasado.

Um NaN nestas colunas significa **"sem atraso desta causa"**, não dado faltante. A imputação correta é com **0**.

In [5]:
delay_cause_cols = ['AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY',
                    'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']

print("NaN antes da imputação:")
print(flights[delay_cause_cols].isnull().sum().to_string())

flights[delay_cause_cols] = flights[delay_cause_cols].fillna(0)

print("\nNaN após imputação:")
print(flights[delay_cause_cols].isnull().sum().to_string())

NaN antes da imputação:
AIR_SYSTEM_DELAY       4665756
SECURITY_DELAY         4665756
AIRLINE_DELAY          4665756
LATE_AIRCRAFT_DELAY    4665756
WEATHER_DELAY          4665756

NaN após imputação:


AIR_SYSTEM_DELAY       0
SECURITY_DELAY         0
AIRLINE_DELAY          0
LATE_AIRCRAFT_DELAY    0
WEATHER_DELAY          0


## Remoção de registros sem DEPARTURE_DELAY

Cerca de 1,5% dos registros não possuem `DEPARTURE_DELAY` — são dados operacionais genuinamente faltantes (voos desviados, falhas de registro). Sem esta informação, não é possível modelar o atraso.

In [6]:
n_before = len(flights)
flights = flights.dropna(subset=["DEPARTURE_DELAY"])
n_after = len(flights)
print(f"Registros removidos: {n_before - n_after:,}")
print(f"Registros retidos: {n_after:,} ({n_after/n_before*100:.1f}%)")
print(f"Shape: {flights.shape}")

Registros removidos: 0
Registros retidos: 5,729,195 (100.0%)
Shape: (5729195, 37)


## Tratamento de NaN em colunas de join

Cerca de 8,3% dos aeroportos no dataset utilizam códigos FAA numéricos (ex: `14747`) que não possuem correspondência na tabela `airports.csv` (códigos IATA de 3 letras). As colunas categóricas resultantes do join ficam com NaN.

Preenchemos com `"Desconhecido"` para permitir operações de groupby sem perda de dados.

In [7]:
join_cat_cols = ['ORIGIN_AIRPORT_NAME', 'ORIGIN_CITY', 'ORIGIN_STATE',
                 'DEST_AIRPORT_NAME', 'DEST_CITY', 'DEST_STATE']

print("NaN antes:")
print(flights[join_cat_cols].isnull().sum().to_string())

flights[join_cat_cols] = flights[join_cat_cols].fillna("Desconhecido")

print("\nNaN após:")
print(flights[join_cat_cols].isnull().sum().to_string())

NaN antes:
ORIGIN_AIRPORT_NAME    483711
ORIGIN_CITY            483711
ORIGIN_STATE           483711
DEST_AIRPORT_NAME      483711
DEST_CITY              483711
DEST_STATE             483711



NaN após:
ORIGIN_AIRPORT_NAME    0
ORIGIN_CITY            0
ORIGIN_STATE           0
DEST_AIRPORT_NAME      0
DEST_CITY              0
DEST_STATE             0


## Feature Engineering

Variáveis derivadas para a modelagem:

| Feature | Derivação | Justificativa |
|---------|-----------|---------------|
| `DEP_HOUR` | `SCHEDULED_DEPARTURE // 100` | Hora de partida agrupada — padrão de atraso varia ao longo do dia |
| `SEASON` | Mapeamento de MONTH → 1-4 | Sazonalidade (inverno/primavera/verão/outono) |
| `IS_WEEKEND` | `DAY_OF_WEEK >= 6` | Padrão de tráfego diferente em fins de semana |
| `IS_DELAYED` | `DEPARTURE_DELAY > 15` | Target binário para classificação (limiar FAA de 15 min) |

In [8]:
# DEP_HOUR
flights['DEP_HOUR'] = flights['SCHEDULED_DEPARTURE'] // 100

# SEASON
season_map = {12: 1, 1: 1, 2: 1, 3: 2, 4: 2, 5: 2, 6: 3, 7: 3, 8: 3, 9: 4, 10: 4, 11: 4}
flights['SEASON'] = flights['MONTH'].map(season_map)

# IS_WEEKEND
flights['IS_WEEKEND'] = (flights['DAY_OF_WEEK'] >= 6).astype(int)

# IS_DELAYED (target)
flights['IS_DELAYED'] = (flights['DEPARTURE_DELAY'] > 15).astype(int)

print("Features criadas:")
print(flights[['MONTH', 'SEASON', 'DAY_OF_WEEK', 'IS_WEEKEND',
               'SCHEDULED_DEPARTURE', 'DEP_HOUR', 'DEPARTURE_DELAY', 'IS_DELAYED']].head(10))

print(f"\nDistribuição IS_DELAYED:")
balance = flights['IS_DELAYED'].value_counts(normalize=True) * 100
print(f"  Pontual (0): {balance[0]:.1f}%")
print(f"  Atrasado (1): {balance[1]:.1f}%")

Features criadas:
   MONTH  SEASON  DAY_OF_WEEK  IS_WEEKEND  SCHEDULED_DEPARTURE  DEP_HOUR  \
0      1       1            4           0                    5         0   
1      1       1            4           0                   10         0   
2      1       1            4           0                   20         0   
3      1       1            4           0                   20         0   
4      1       1            4           0                   25         0   
5      1       1            4           0                   25         0   
6      1       1            4           0                   25         0   
7      1       1            4           0                   30         0   
8      1       1            4           0                   30         0   
9      1       1            4           0                   30         0   

   DEPARTURE_DELAY  IS_DELAYED  
0            -11.0           0  
1             -8.0           0  
2             -2.0           0  
3            

## Verificação final

Conferir shape, tipos e ausência de nulls em colunas críticas antes da exportação.

In [9]:
print(f"Shape final: {flights.shape}")
print(f"\nColunas: {list(flights.columns)}")
print(f"\nValores ausentes em colunas críticas:")
critical_cols = ['DEPARTURE_DELAY', 'IS_DELAYED', 'MONTH', 'DAY_OF_WEEK',
                 'DEP_HOUR', 'SEASON', 'IS_WEEKEND', 'DISTANCE', 'AIRLINE']
print(flights[critical_cols].isnull().sum().to_string())
print(f"\nDescrição do dataset limpo:")
flights.describe()

Shape final: (5729195, 41)

Colunas: ['MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'WHEELS_ON', 'TAXI_IN', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME', 'ARRIVAL_DELAY', 'DIVERTED', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY', 'AIRLINE_NAME', 'ORIGIN_AIRPORT_NAME', 'ORIGIN_CITY', 'ORIGIN_STATE', 'ORIGIN_LAT', 'ORIGIN_LON', 'DEST_AIRPORT_NAME', 'DEST_CITY', 'DEST_STATE', 'DEST_LAT', 'DEST_LON', 'DEP_HOUR', 'SEASON', 'IS_WEEKEND', 'IS_DELAYED']

Valores ausentes em colunas críticas:
DEPARTURE_DELAY    0
IS_DELAYED         0
MONTH              0
DAY_OF_WEEK        0
DEP_HOUR           0
SEASON             0
IS_WEEKEND         0
DISTANCE           0
AIRLINE            0

Descrição do dataset limpo:


,MONTH,DAY,DAY_OF_WEEK,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,ORIGIN_LAT,ORIGIN_LON,DEST_LAT,DEST_LON,DEP_HOUR,SEASON,IS_WEEKEND,IS_DELAYED
count,5.729195e+06,5.729195e+06,5.729195e+06,5.729195e+06,5.729195e+06,5.729195e+06,5.729195e+06,5.729195e+06,5.729194e+06,5.714008e+06,5.714008e+06,5.729195e+06,5.726566e+06,5.726566e+06,5.729195e+06,5.726566e+06,5.714008e+06,5.729195e+06,5.729195e+06,5.729195e+06,5.729195e+06,5.729195e+06,5.729195e+06,5.240912e+06,5.240912e+06,5.240900e+06,5.240900e+06,5.729195e+06,5.729195e+06,5.729195e+06,5.729195e+06
mean,6.547562e+00,1.570848e+01,3.932479e+00,1.328890e+03,1.335102e+03,9.338837e+00,1.607112e+01,1.357142e+03,1.419480e+02,1.370062e+02,1.135116e+02,8.248575e+02,1.471469e+03,7.434971e+00,1.493315e+03,1.476491e+03,4.407057e+00,2.650809e-03,2.502230e+00,1.413549e-02,3.521080e+00,4.356970e+00,5.411289e-01,3.662367e+01,-9.556861e+01,3.662358e+01,-9.556906e+01,1.302202e+01,2.523329e+00,2.611877e-01,1.774309e-01
std,3.397027e+00,8.774635e+00,1.986101e+00,4.834692e+02,4.963902e+02,3.699246e+01,8.894483e+00,4.979969e+02,7.533501e+01,7.421107e+01,7.223082e+01,6.087992e+02,5.221879e+02,5.638548e+00,5.068360e+02,5.263197e+02,3.927130e+01,5.141772e-02,1.315419e+01,9.239484e-01,2.202142e+01,2.072791e+01,8.876031e+00,6.009700e+00,1.818742e+01,6.011084e+00,1.819061e+01,4.821875e+00,1.099398e+00,4.392820e-01,3.820330e-01
min,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,-8.200000e+01,1.000000e+00,1.000000e+00,1.800000e+01,1.400000e+01,7.000000e+00,3.100000e+01,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,-8.700000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.348345e+01,-1.766460e+02,1.348345e+01,-1.766460e+02,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00
25%,4.000000e+00,8.000000e+00,2.000000e+00,9.160000e+02,9.210000e+02,-5.000000e+00,1.100000e+01,9.350000e+02,8.600000e+01,8.200000e+01,6.000000e+01,3.730000e+02,1.054000e+03,4.000000e+00,1.110000e+03,1.059000e+03,-1.300000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,3.289595e+01,-1.119778e+02,3.289595e+01,-1.119778e+02,9.000000e+00,2.000000e+00,0.000000e+00,0.000000e+00
50%,7.000000e+00,1.600000e+01,4.000000e+00,1.325000e+03,1.330000e+03,-2.000000e+00,1.400000e+01,1.343000e+03,1.230000e+02,1.180000e+02,9.400000e+01,6.500000e+02,1.509000e+03,6.000000e+00,1.520000e+03,1.512000e+03,-5.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,3.724433e+01,-9.025803e+01,3.724433e+01,-9.025803e+01,1.300000e+01,3.000000e+00,0.000000e+00,0.000000e+00
75%,9.000000e+00,2.300000e+01,6.000000e+00,1.730000e+03,1.740000e+03,7.000000e+00,1.900000e+01,1.754000e+03,1.740000e+02,1.680000e+02,1.440000e+02,1.066000e+03,1.911000e+03,9.000000e+00,1.917000e+03,1.917000e+03,8.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4.078839e+01,-8.168786e+01,4.078839e+01,-8.168786e+01,1.700000e+01,3.000000e+00,1.000000e+00,0.000000e+00
max,1.200000e+01,3.100000e+01,7.000000e+00,2.359000e+03,2.400000e+03,1.988000e+03,2.250000e+02,2.400000e+03,7.180000e+02,7.660000e+02,6.900000e+02,4.983000e+03,2.400000e+03,2.480000e+02,2.400000e+03,2.400000e+03,1.971000e+03,1.000000e+00,1.134000e+03,5.730000e+02,1.971000e+03,1.331000e+03,1.211000e+03,7.128545e+01,-6.479856e+01,7.128545e+01,-6.479856e+01,2.300000e+01,4.000000e+00,1.000000e+00,1.000000e+00


## Exportação em Parquet

Salvamos o dataset limpo em formato Parquet (compressão snappy) para evitar reprocessamento nos notebooks seguintes. O Parquet preserva os tipos de dados e é significativamente mais rápido para leitura.

In [10]:
output_path = "../data/flights_clean.parquet"
flights.to_parquet(output_path, index=False)

# Verificação de roundtrip
df_check = pd.read_parquet(output_path)
assert df_check.shape == flights.shape, f"Shape mismatch: {df_check.shape} vs {flights.shape}"

import os
file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"Arquivo salvo: {output_path}")
print(f"Tamanho: {file_size_mb:.1f} MB")
print(f"Shape verificada: {df_check.shape}")
print("\nExportação concluída com sucesso!")

Arquivo salvo: ../data/flights_clean.parquet
Tamanho: 168.5 MB
Shape verificada: (5729195, 41)

Exportação concluída com sucesso!
